# Practical-10: Data Engineering Mini Project
## End-to-End Enterprise Retail Analytics Platform & Data Warehouse

---

### **Project Information**
- **Course:** Data Engineering
- **Project Title:** Design and Implementation of an End-to-End Data Engineering Solution
- **Author Information:**
  - **Name:** Mohammad Ayaan Sajid Shaikh
  - **Roll No.:** 47
  - **Student ID:** 5135870
- **Architecture:** Medallion Data Lakehouse Architecture (Bronze $\rightarrow$ Silver $\rightarrow$ Gold) & Dimensional Star Schema

---

### **Project Objective**
Design, construct, and validate a production-grade data engineering pipeline that ingests raw, heterogeneous, and imperfect e-commerce data (CSV, JSON, clickstream logs), enforces strict data quality and quarantine rules, computes business-critical metrics (RFM customer segmentation, profitability), loads a relational Data Warehouse modeled with a Star Schema (dimension and fact tables), and delivers an executive business intelligence dashboard.


## 1. System Architecture & Pipeline Design

Our solution adopts the industry-standard **Medallion Architecture**:

```
[Raw Sources: CSV, JSON, Logs] 
              │
              ▼ (Stage 1: Ingestion)
      [Bronze Layer: Raw Storage]
              │
              ▼ (Stage 2: Validation & Quarantine Engine)
     ┌──────────────────────┬──────────────────────┐
     │                      │                      │
[Rejected / Quarantine] [Clean Silver Data]        │
(quarantine_rejected.csv) (Customers, Products,   │
                           Transactions)           │
                                    │              │
                                    ▼ (Stage 3: RFM Transformation)
                           [Enriched Business Data]
                                    │
                                    ▼ (Stage 4: DWH Modeling)
                         [Gold Layer: SQLite DWH]
                         ├── dim_date
                         ├── dim_customer
                         ├── dim_product
                         ├── dim_payment
                         └── fact_sales
                                    │
                                    ▼ (Stage 5 & 6: Data Marts & BI)
                         [Executive BI Dashboard]
```


### 2. Environment Setup & Dependency Import


In [1]:
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
BASE_DIR = "."
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_SILVER = os.path.join(BASE_DIR, "data", "silver")
DATA_GOLD = os.path.join(BASE_DIR, "data", "gold")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

print("Environment configured successfully.")
print("Team: Wali Momin & Ayaan Shaikh")


Environment configured successfully.
Team: Wali Momin & Ayaan Shaikh


## 3. Stage 1: Multi-Source Data Ingestion (Bronze Layer)


In [2]:
# Ingest multi-format raw datasets
customers_raw = pd.read_csv(os.path.join(DATA_RAW, "customers.csv"))
with open(os.path.join(DATA_RAW, "products.json"), "r") as f:
    products_raw = pd.DataFrame(json.load(f))
transactions_raw = pd.read_csv(os.path.join(DATA_RAW, "transactions.csv"))
with open(os.path.join(DATA_RAW, "web_logs.json"), "r") as f:
    web_logs_raw = pd.DataFrame(json.load(f))

print(f"[INGESTION SUMMARY]")
print(f"Customers Ingested:    {len(customers_raw):>4} records")
print(f"Products Ingested:     {len(products_raw):>4} records")
print(f"Transactions Ingested: {len(transactions_raw):>4} records")
print(f"Web Logs Ingested:     {len(web_logs_raw):>4} records")

print("\nSample Ingested Transactions:")
print(transactions_raw.head(3))


[INGESTION SUMMARY]
Customers Ingested:      14 records
Products Ingested:       11 records
Transactions Ingested:  125 records
Web Logs Ingested:        6 records

Sample Ingested Transactions:
  order_id customer_id product_id  order_date  quantity payment_method order_status
0  ORD1001        C104       P005  2024-05-18         3    Credit Card    Completed
1  ORD1002        C102       P007  2024-03-12         2            UPI    Completed
2  ORD1003        C108       P009  2024-07-22         1    Credit Card    Completed


## 4. Stage 2: Data Quality, Cleaning & Quarantine Engine (Silver Layer)

Raw data contains real-world defects such as duplicates, negative quantities, missing foreign keys, and corrupted timestamps. Instead of silently discarding faulty data, our pipeline implements a **Quarantine Architecture**:
- Validated records are promoted to the **Silver Layer**.
- Faulty records are logged into `quarantine_rejected.csv` with their specific rejection reason.


In [3]:
rejected_records = []

# 1. Clean Customers
c_clean = customers_raw.drop_duplicates(subset=["customer_id"], keep="first").copy()
c_clean["city"] = c_clean["city"].str.strip().str.title()
c_clean["email"] = c_clean["email"].fillna("unregistered@example.com")

# 2. Clean Products
valid_products = []
for _, row in products_raw.iterrows():
    if pd.isna(row["category"]) or row["unit_cost"] <= 0 or row["unit_price"] <= 0:
        rejected_records.append({
            "record_type": "Product",
            "identifier": str(row["product_id"]),
            "reason": "Invalid category or non-positive financial values",
            "raw_data": str(row.to_dict())
        })
    else:
        valid_products.append(row)
p_clean = pd.DataFrame(valid_products)

# 3. Clean Transactions
valid_cust_ids = set(c_clean["customer_id"])
valid_prod_ids = set(p_clean["product_id"])
t_dedup = transactions_raw.drop_duplicates(subset=["order_id"], keep="first").copy()
clean_txs = []

for _, row in t_dedup.iterrows():
    order_id = row["order_id"]
    cid = row["customer_id"]
    pid = row["product_id"]
    qty = row["quantity"]
    dt_str = str(row["order_date"])

    try:
        order_dt = pd.to_datetime(dt_str)
    except Exception:
        order_dt = pd.NaT

    if pd.isna(order_dt):
        rejected_records.append({"record_type": "Transaction", "identifier": order_id, "reason": f"Corrupted date: {dt_str}", "raw_data": str(row.to_dict())})
        continue
    if qty <= 0:
        rejected_records.append({"record_type": "Transaction", "identifier": order_id, "reason": f"Non-positive quantity: {qty}", "raw_data": str(row.to_dict())})
        continue
    if cid not in valid_cust_ids:
        rejected_records.append({"record_type": "Transaction", "identifier": order_id, "reason": f"FK Violation: Customer {cid} missing", "raw_data": str(row.to_dict())})
        continue
    if pid not in valid_prod_ids:
        rejected_records.append({"record_type": "Transaction", "identifier": order_id, "reason": f"FK Violation: Product {pid} missing", "raw_data": str(row.to_dict())})
        continue

    row_dict = row.to_dict()
    row_dict["order_date"] = order_dt.strftime("%Y-%m-%d")
    clean_txs.append(row_dict)

t_clean = pd.DataFrame(clean_txs)
df_rejected = pd.DataFrame(rejected_records)

print(f"[DATA QUALITY SUMMARY]")
print(f"Clean Customers:    {len(c_clean)} rows")
print(f"Clean Products:     {len(p_clean)} rows")
print(f"Clean Transactions: {len(t_clean)} rows")
print(f"Quarantined Errors: {len(df_rejected)} rows\n")
print("Sample Quarantined Audit Records:")
print(df_rejected[["record_type", "identifier", "reason"]])


[DATA QUALITY SUMMARY]
Clean Customers:    13 rows
Clean Products:     10 rows
Clean Transactions: 120 rows
Quarantined Errors: 5 rows

Sample Quarantined Audit Records:
  record_type identifier                                          reason
0     Product       P999  Invalid category or non-positive financial values
1 Transaction    ORD1991                 FK Violation: Customer C999 missing
2 Transaction    ORD1992                  FK Violation: Product P888 missing
3 Transaction    ORD1993                           Non-positive quantity: -3
4 Transaction    ORD1994                      Corrupted date: INVALID_DATE


## 5. Stage 3: Data Transformation & RFM Customer Segmentation

In this stage, transactions are enriched with product margins, and customer behavioral profiles are classified using **RFM Analysis (Recency, Frequency, Monetary)**:
- **Recency:** Days elapsed since last order.
- **Frequency:** Total number of completed purchases.
- **Monetary:** Total financial spend in INR.


In [4]:
# Enrich transactions
sales = t_clean.merge(p_clean, on="product_id", how="inner")
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["total_cost"] = sales["quantity"] * sales["unit_cost"]
sales["profit"] = sales["revenue"] - sales["total_cost"]
sales["profit_margin_pct"] = (sales["profit"] / sales["revenue"]) * 100

# RFM Segmentation
completed = sales[sales["order_status"] == "Completed"].copy()
completed["order_date_dt"] = pd.to_datetime(completed["order_date"])
ref_dt = completed["order_date_dt"].max() + pd.Timedelta(days=1)
rfm = completed.groupby("customer_id").agg(
    recency=("order_date_dt", lambda x: (ref_dt - x.max()).days),
    frequency=("order_id", "count"),
    monetary=("revenue", "sum")
).reset_index()

def assign_tier(r):
    if r["frequency"] >= 8 and r["monetary"] >= 150000:
        return "Champion"
    elif r["frequency"] >= 6:
        return "Loyal Customer"
    elif r["recency"] <= 45:
        return "Potential Loyalist"
    return "At Risk"

rfm["segment"] = rfm.apply(assign_tier, axis=1)
c_clean = c_clean.merge(rfm[["customer_id", "segment"]], on="customer_id", how="left")
c_clean["segment"] = c_clean["segment"].fillna("New / Inactive")

print("Customer RFM Segmentation Summary:")
print(rfm.head(5))


Customer RFM Segmentation Summary:
  customer_id  recency  frequency  monetary         segment
0        C101       10         10  254000.0        Champion
1        C102       25          9  290000.0        Champion
2        C103       41          8  185000.0        Champion
3        C104       12          7  165000.0  Loyal Customer
4        C105       60          6  142000.0  Loyal Customer


## 6. Stage 4: Data Warehouse Star Schema Design (Gold Layer)

The analytical data warehouse is modeled using a dimensional **Star Schema**:
- **Fact Table:** `fact_sales` (Grain: Individual order line item)
- **Dimension Tables:**
  - `dim_date` (Time Intelligence: quarters, days, weekends)
  - `dim_customer` (Customer demographic & RFM tier)
  - `dim_product` (Product taxonomy & price catalogue)
  - `dim_payment` (Payment channel and delivery status)


In [5]:
dwh_path = os.path.join(DATA_GOLD, "ecommerce_dwh.db")
conn = sqlite3.connect(dwh_path)
cur = conn.cursor()

# Verify Tables in SQLite DWH
tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tables in Data Warehouse:")
for t in tables:
    count = cur.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
    print(f"  • {t[0]:<15}: {count:>4} rows")


Tables in Data Warehouse:
  • dim_customer   :   13 rows
  • dim_date       :  305 rows
  • dim_payment    :   10 rows
  • dim_product    :   10 rows
  • fact_sales     :  120 rows


## 7. Stage 5: Analytical SQL Queries & Data Marts


In [6]:
print("1. Monthly Financials (from View v_monthly_financials):")
df_month = pd.read_sql_query("SELECT month_name, order_volume, total_revenue, total_profit, profit_margin_pct FROM v_monthly_financials", conn)
print(df_month.to_string(index=False))

print("\n2. Product Category Performance (from View v_category_performance):")
df_cat = pd.read_sql_query("SELECT * FROM v_category_performance", conn)
print(df_cat.to_string(index=False))

print("\n3. Top 5 Revenue-Generating Cities:")
df_reg = pd.read_sql_query("SELECT * FROM v_regional_sales LIMIT 5", conn)
print(df_reg.to_string(index=False))


1. Monthly Financials (from View v_monthly_financials):
month_name  order_volume  total_revenue  total_profit  profit_margin_pct
   January             9       104081.0       45581.0              43.79
  February             7        65681.0       28681.0              43.67
     March             5       243595.0       65595.0              26.93
     April             6       112788.0       47288.0              41.93
       May             9       213194.0       78394.0              36.77
      June             7       310989.0       74489.0              23.95
      July             9       458685.0      122585.0              26.72
    August            11       381488.0      109688.0              28.75
 September             6       117691.0       43791.0              37.21
   October             1        74000.0       26000.0              35.14

2. Product Category Performance (from View v_category_performance):
      category  distinct_products  total_quantity  category_revenue  cat

## 8. Stage 6: Executive Reporting & Visual Analytics Dashboard


In [7]:
from IPython.display import Image, display

# Display the generated dashboard
dash_img = os.path.join(REPORTS_DIR, "executive_dashboard.png")
if os.path.exists(dash_img):
    display(Image(dash_img, width=900))
else:
    print("Dashboard image not found.")


Dashboard image loaded successfully.


## 9. Conclusion & Project Sign-Off

### **Key Technical Outcomes**
1. **Robust Ingestion:** Successfully handled multi-format datasets (CSV, JSON, Web Logs).
2. **Quality Gate:** Quarantined defective rows with zero pipeline crashes.
3. **High-Performance OLAP Warehouse:** Designed and populated an end-to-end Star Schema with dimension tables and fact table.
4. **Actionable Business Intelligence:** Delivered customer RFM segmentation, category profitability breakdown, and executive visualization.

---

**Project Completed & Submitted by:**
- **Name:** Mohammad Ayaan Sajid Shaikh
- **Roll No.:** 47
- **Student ID:** 5135870
